In [1]:
import os
import shutil
import random
from sklearn.model_selection import train_test_split
from google.colab import drive
import tensorflow as tf
import numpy as np
drive.mount('/content/drive')

base_dir = '/content/drive/My Drive/Datasets/Brain_Tumor_MRI'
train_dir = os.path.join(base_dir, 'Training')
saved_model = os.path.join(base_dir, 'saved_models/BrainTumorMRI_VGG16_26102025_0850_model.keras')


Mounted at /content/drive


In [2]:
# @title
# import tensorflow as tf
# import tensorflow_addons as tfa  # optional, untuk Gaussian blur atau interpolasi
# import numpy as np

# def nlm_denoise_bpda(x: tf.Tensor, h: float = 10.0, patch_size: int = 3, window_size: int = 7) -> tf.Tensor:
#     """
#     Apply Non-Local Means (NLM) denoising on input tensor using BPDA approximation.

#     Forward pass: applies standard NLM (non-differentiable, patch averaging)
#     Backward pass: identity gradient (BPDA trick) so gradient can flow for adversarial training.

#     Parameters
#     ----------
#     x : tf.Tensor
#         Input image tensor in shape (B,H,W,C), dtype=float32, values in [0,255]
#     h : float
#         Filtering parameter controlling decay of weights based on patch similarity
#     patch_size : int
#         Size of the patch used for similarity comparison
#     window_size : int
#         Search window size for computing non-local averages

#     Returns
#     -------
#     tf.Tensor
#         Denoised image tensor, same shape as input
#     """

#     @tf.custom_gradient
#     def _forward_nlm(x_inner):
#         """
#         Forward NLM approximation: non-differentiable patch-based averaging
#         Backward gradient: identity (BPDA)
#         """
#         # Convert to numpy for patch processing (Buades et al., 2005)
#         x_np = x_inner.numpy()
#         B, H, W, C = x_np.shape
#         x_denoised = np.empty_like(x_np)

#         for b in range(B):
#             for c in range(C):
#                 img = x_np[b, :, :, c]
#                 pad_size = window_size // 2 + patch_size // 2
#                 img_pad = np.pad(img, pad_size, mode='reflect')
#                 denoised = np.zeros_like(img)
#                 for i in range(H):
#                     for j in range(W):
#                         i1 = i + pad_size
#                         j1 = j + pad_size
#                         patch = img_pad[i1 - patch_size//2:i1 + patch_size//2 + 1,
#                                         j1 - patch_size//2:j1 + patch_size//2 + 1]

#                         # Search window
#                         win = img_pad[i1 - window_size//2:i1 + window_size//2 + 1,
#                                       j1 - window_size//2:j1 + window_size//2 + 1]

#                         # Compute weight (simplified, Gaussian of L2 distance)
#                         weights = np.exp(- ((win - patch.mean())**2) / (h**2))
#                         denoised[i, j] = np.sum(weights * win) / (np.sum(weights) + 1e-12)
#                 x_denoised[b, :, :, c] = denoised

#         x_denoised_tf = tf.convert_to_tensor(x_denoised, dtype=tf.float32)

#         def grad(dy):
#             # BPDA trick: pass gradient as identity
#             return dy

#         return x_denoised_tf, grad

#     return _forward_nlm(x)


In [3]:
# @title
# from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess

# def preprocess_with_nlm(x_raw: tf.Tensor) -> tf.Tensor:
#     """
#     Preprocess raw [0,255] MRI images for VGG16 with optional NLM denoising.

#     Parameters
#     ----------
#     x_raw : tf.Tensor
#         Raw images, shape (B,H,W,C), dtype=float32, values [0,255]

#     Returns
#     -------
#     tf.Tensor
#         Preprocessed images ready for VGG16
#     """
#     # --- 1) NLM denoise (BPDA) ---
#     x_denoised = nlm_denoise_bpda(x_raw, h=10.0, patch_size=3, window_size=7)

#     # --- 2) VGG16 preprocessing ---
#     x_preprocessed = vgg_preprocess(x_denoised)

#     return x_preprocessed


In [4]:
from typing import Callable, Optional
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess

EPS = 1e-12

@tf.custom_gradient
def nlm_denoise_bpda_gpu_subsampled(
    x: tf.Tensor,
    h: float = 0.1,
    patch_size: int = 3,
    window_size: int = 7,
    step: int = 2  # subsampling stride, 1=full NLM, 2=faster
) -> tf.Tensor:
    assert patch_size % 2 == 1 and window_size % 2 == 1
    x_dtype = x.dtype
    x_f = tf.cast(x, tf.float32) / 255.0

    pad_r = (window_size - 1) // 2
    x_f = tf.pad(x_f, [[0,0],[pad_r,pad_r],[pad_r,pad_r],[0,0]], mode='REFLECT')

    B, H, W, C = tf.unstack(tf.shape(x_f))
    k = patch_size
    D = k * k * C

    patches = tf.image.extract_patches(
        images=x_f,
        sizes=[1, k, k, 1],
        strides=[1, 1, 1, 1],
        rates=[1, 1, 1, 1],
        padding='SAME'
    )
    patches = tf.reshape(patches, [B, H, W, D])
    center = patches

    weighted_sum = tf.zeros_like(x_f)
    weight_sum = tf.zeros([B, H, W, 1], dtype=tf.float32)
    h_sq = tf.cast(h ** 2, tf.float32)

    # subsampled loop over neighborhood
    for dy in tf.range(-pad_r, pad_r + 1, step):
        for dx in tf.range(-pad_r, pad_r + 1, step):
            patches_shifted = tf.roll(patches, shift=[dy, dx], axis=[1, 2])
            img_shifted = tf.roll(x_f, shift=[dy, dx], axis=[1, 2])

            diff = center - patches_shifted
            dist2 = tf.reduce_sum(tf.square(diff), axis=-1, keepdims=True)

            w = tf.exp(-dist2 / (h_sq + 1e-12))
            weighted_sum += w * img_shifted
            weight_sum += w

    denoised = weighted_sum / (weight_sum + 1e-12)
    denoised = denoised[:, pad_r:-pad_r, pad_r:-pad_r, :]
    denoised_scaled = tf.clip_by_value(denoised * 255.0, 0.0, 255.0)
    denoised_scaled = tf.cast(denoised_scaled, x_dtype)

    # GRAD: return single grad (upstream) because only `x` is tensor input in your calls
    def grad(upstream):
        return upstream

    return denoised_scaled, grad


def preprocess_with_nlm_gpu(
    x_raw: tf.Tensor,
    use_nlm: bool = True,
    h: float = 0.1,
    patch_size: int = 3,
    window_size: int = 7,
    step: int = 2,  #subsampling
    vgg_preprocess_fn: Optional[Callable[[tf.Tensor], tf.Tensor]] = None,
) -> tf.Tensor:
    x = tf.cast(x_raw, tf.float32)
    if use_nlm:
        x = nlm_denoise_bpda_gpu_subsampled(
            x, h=h, patch_size=patch_size,
            window_size=window_size, step=step
        )
    if vgg_preprocess_fn is not None:
        x = vgg_preprocess_fn(x)
    return x


In [5]:
import tensorflow as tf
from typing import Literal, Callable

EPS = 1e-12

def _kl_divergence_per_sample(logits_p, logits_q):
    """KL(p || q) per-sample (sum over classes) -> shape (batch,)."""
    p = tf.nn.softmax(logits_p, axis=-1)
    q = tf.nn.softmax(logits_q, axis=-1)
    p = tf.clip_by_value(p, 1e-12, 1.0)
    q = tf.clip_by_value(q, 1e-12, 1.0)
    kl = tf.reduce_sum(p * (tf.math.log(p) - tf.math.log(q)), axis=-1)
    return kl

def trades_loss_for_vgg(
    model: tf.keras.Model,
    x_natural_raw: tf.Tensor,
    y: tf.Tensor,
    optimizer: tf.keras.optimizers.Optimizer,
    step_size: float = 2.0,
    epsilon: float = 8.0,
    perturb_steps: int = 10,
    beta: float = 1.0,
    distance: Literal["l_inf", "l_2"] = "l_inf",
    preprocess_fn: Callable[[tf.Tensor], tf.Tensor] = None,
):
    """
    TRADES loss for models expecting VGG preprocess. Works on raw pixel inputs [0,255].
    - x_natural_raw: float32 tensor, values in [0,255]
    - step_size, epsilon: pixel units (e.g., 2.0, 8.0)
    - preprocess_fn: function that maps raw pixels -> model input (e.g. vgg16.preprocess_input)
                     Must be applied INSIDE gradient tapes when computing logits from x_adv.
    Returns scalar loss tensor; also applies optimizer update to model weights.
    """

    x_natural_raw = tf.cast(x_natural_raw, tf.float32)
    y = tf.cast(tf.reshape(y, (-1,)), tf.int32)
    batch_size = tf.shape(x_natural_raw)[0]

    # --- 1) create x_adv in raw pixel space ---
    x_adv = x_natural_raw + 0.001 * tf.random.normal(tf.shape(x_natural_raw), dtype=tf.float32)
    x_adv = tf.clip_by_value(x_adv, 0.0, 255.0)

    # compute logits_nat once (preprocess then model), detach
    inp_nat_for_model = preprocess_fn(x_natural_raw) if preprocess_fn is not None else x_natural_raw
    logits_nat = model(inp_nat_for_model, training=False)
    logits_nat = tf.stop_gradient(logits_nat)

    if distance == "l_inf":
        for _ in range(int(perturb_steps)):
            x_adv_var = tf.Variable(x_adv)
            with tf.GradientTape() as tape:
                tape.watch(x_adv_var)
                inp_adv = preprocess_fn(x_adv_var) if preprocess_fn is not None else x_adv_var
                logits_adv = model(inp_adv, training=False)
                kl_per = _kl_divergence_per_sample(logits_nat, logits_adv)
                loss_kl = tf.reduce_mean(kl_per)
            grad = tape.gradient(loss_kl, x_adv_var)
            x_adv = x_adv + step_size * tf.sign(grad)
            x_adv = tf.clip_by_value(x_adv, x_natural_raw - epsilon, x_natural_raw + epsilon)
            x_adv = tf.clip_by_value(x_adv, 0.0, 255.0)
        x_adv = tf.stop_gradient(x_adv)

    elif distance == "l_2":
        delta = x_adv - x_natural_raw
        delta = tf.Variable(delta)
        for _ in range(int(perturb_steps)):
            with tf.GradientTape() as tape:
                tape.watch(delta)
                adv = x_natural_raw + delta
                adv = tf.clip_by_value(adv, 0.0, 255.0)
                inp_adv = preprocess_fn(adv) if preprocess_fn is not None else adv
                logits_adv = model(inp_adv, training=False)
                kl_per = _kl_divergence_per_sample(logits_nat, logits_adv)
                loss_kl = tf.reduce_mean(kl_per)
            g = tape.gradient(loss_kl, delta)
            # normalize per-sample
            g_flat = tf.reshape(g, [tf.shape(g)[0], -1])
            g_norm = tf.norm(g_flat, axis=1, keepdims=True)
            g_norm_safe = tf.maximum(g_norm, EPS)
            g_unit = g / tf.reshape(g_norm_safe, tf.concat([[tf.shape(g)[0]], tf.ones(tf.rank(g)-1, tf.int32)], axis=0))
            delta.assign_add(step_size * g_unit)
            # project to L2 ball
            delta_flat = tf.reshape(delta, [tf.shape(delta)[0], -1])
            delta_norm = tf.norm(delta_flat, axis=1, keepdims=True)
            factor = tf.minimum(1.0, epsilon / tf.maximum(delta_norm, EPS))
            delta.assign(tf.reshape(delta_flat * factor, tf.shape(delta)))
            delta.assign(tf.clip_by_value(x_natural_raw + delta, 0.0, 255.0) - x_natural_raw)
        x_adv = tf.stop_gradient(x_natural_raw + delta)
    else:
        x_adv = tf.clip_by_value(x_adv, 0.0, 255.0)
        x_adv = tf.stop_gradient(x_adv)

    # --- 2) compute TRADES loss and update model ---
    with tf.GradientTape() as tape:
        inp_nat = preprocess_fn(x_natural_raw) if preprocess_fn is not None else x_natural_raw
        inp_adv = preprocess_fn(x_adv) if preprocess_fn is not None else x_adv

        logits = model(inp_nat, training=True)
        logits_adv = model(inp_adv, training=True)

        loss_natural = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=logits))
        kl_per = _kl_divergence_per_sample(logits, logits_adv)
        loss_robust = tf.reduce_mean(kl_per)
        loss = loss_natural + beta * loss_robust

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    return loss


In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess


# Generator *tanpa* preprocess_input -> returns raw pixels in [0,255]
train_datagen_raw = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    shear_range=0.1,
    brightness_range=[0.90, 1.10],
    zoom_range=0.1,
    horizontal_flip=False,
    validation_split=0.1
)

train_generator_raw = train_datagen_raw.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    subset='training',
    class_mode='sparse'
)

val_datagen_raw = ImageDataGenerator(validation_split=0.1)
val_generator_raw = val_datagen_raw.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='sparse',
    subset='validation',
    shuffle=False
)

Found 5143 images belonging to 4 classes.
Found 569 images belonging to 4 classes.


In [7]:
# load model
model = tf.keras.models.load_model(saved_model, compile=False)
optimizer = tf.keras.optimizers.SGD(learning_rate=1e-3, momentum=0.9)

preprocess_fn = lambda x: preprocess_with_nlm_gpu(
    x,
    use_nlm=True,
    h=10.0,
    patch_size=3,
    window_size=7,
    vgg_preprocess_fn=vgg_preprocess
)

In [8]:
checkpoint_TRADES_model_dir = os.path.join(
    base_dir, 'saved_models/Checkpoint_TRADES_trained_vgg16_model_31102025_1434.keras'
)
final_TRADES_model_dir = os.path.join(
    base_dir, 'saved_models/FULL_TRADES_trained_vgg16_model_31102025_1434.keras'
)

# training hyperparams
epochs = 100
beta = 5
step_size = 1     # pixels
epsilon = 4        # pixels
perturb_steps = 10

patience = 5
best_val_loss = float('inf')
wait = 0
val_subset_size = 64
val_batch_size = 8

In [ ]:
from tqdm import tqdm
import numpy as np
import tensorflow as tf

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    batch_losses = []

    train_pbar = tqdm(enumerate(train_generator_raw), total=len(train_generator_raw), desc="Training", ncols=100)

    for step, (x_batch_raw, y_batch) in train_pbar:
        x_batch_raw = tf.cast(x_batch_raw, tf.float32)

        loss = trades_loss_for_vgg(
            model=model,
            x_natural_raw=x_batch_raw,
            y=y_batch,
            optimizer=optimizer,
            step_size=step_size,
            epsilon=epsilon,
            perturb_steps=perturb_steps,
            beta=beta,
            distance="l_inf",
            preprocess_fn=preprocess_fn
        )

        batch_losses.append(float(loss.numpy()))

        train_pbar.set_postfix({"loss": f"{np.mean(batch_losses[-10:]):.4f}"})

    train_loss = np.mean(batch_losses)
    print(f"Epoch {epoch+1} mean training loss: {train_loss:.4f}")

    val_losses = []
    val_sample_count = 0

    val_pbar = tqdm(enumerate(val_generator_raw), total=min(len(val_generator_raw), val_subset_size // val_generator_raw.batch_size), desc="Validation", ncols=100)

    for step, (x_val, y_val) in val_pbar:
        x_val = tf.cast(x_val, tf.float32)
        preds = model(preprocess_fn(x_val), training=False)
        val_loss = tf.keras.losses.categorical_crossentropy(y_val, preds)
        val_losses.append(np.mean(val_loss.numpy()))

        val_sample_count += len(x_val)
        val_pbar.set_postfix({"val_loss": f"{np.mean(val_losses):.4f}"})

        if val_sample_count >= val_subset_size:
            break

    val_loss_mean = np.mean(val_losses)
    print(f"Validation loss (sampled): {val_loss_mean:.4f}")

    if val_loss_mean < best_val_loss:
        print(f"Validation loss improved from {best_val_loss:.4f} → {val_loss_mean:.4f}")
        best_val_loss = val_loss_mean
        wait = 0
        model.save(checkpoint_TRADES_model_dir)
        print(f"✅ Checkpoint saved at: {checkpoint_TRADES_model_dir}")
    else:
        wait += 1
        print(f"No improvement from {best_val_loss:.4f}. Patience counter: {wait}/{patience}")

    if wait >= patience:
        print("⏹ Early stopping triggered — no improvement for several epochs.")
        break

print(f"\nTraining done. Saved final model to: {final_TRADES_model_dir}")
model.save(final_TRADES_model_dir)



Epoch 1/100
  Step 10: loss 1.7227
  Step 20: loss 1.7165
  Step 30: loss 1.6919
  Step 40: loss 1.6125
  Step 50: loss 1.6248
  Step 60: loss 1.6326
  Step 70: loss 1.6497
  Step 80: loss 1.5843
  Step 90: loss 1.6223
  Step 100: loss 1.6092
  Step 110: loss 1.6463
  Step 120: loss 1.6521
  Step 130: loss 1.5490


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input

FGSM_advx_dir = os.path.join(base_dir, 'Adversarial_Example/Testing_FGSM_Adversarial_Example')
PGD_advx_dir = os.path.join(base_dir, 'Adversarial_Example/Testing_PGD_Adversarial_Example')
raw_test_dir = os.path.join(base_dir, 'Testing')

FGSM_advx_datagen = ImageDataGenerator()
FGSM_advx_generator = FGSM_advx_datagen.flow_from_directory(
    FGSM_advx_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='sparse',
    shuffle=False
)

PGD_advx_datagen = ImageDataGenerator()
PGD_advx_generator = PGD_advx_datagen.flow_from_directory(
    PGD_advx_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='sparse',
    shuffle=False
)

raw_test_datagen = ImageDataGenerator()
raw_test_generator = raw_test_datagen.flow_from_directory(
    raw_test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='sparse',
    shuffle=False
)


In [ ]:
trained_model = final_TRADES_model_dir
model = tf.keras.models.load_model(trained_model)

In [ ]:
# @title
# Compile ulang untuk evaluasi
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(),
#     loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
#     metrics=['accuracy']
# )

In [ ]:
from sklearn.metrics import classification_report
# Evaluate on test (apply preprocess before model)
y_true, y_pred = [], []
for x_batch_raw, y_batch in FGSM_advx_generator:
    x_in = preprocess_fn(tf.cast(x_batch_raw, tf.float32))
    preds = model.predict(x_in, verbose=0)
    y_true.extend(y_batch)
    y_pred.extend(np.argmax(preds, axis=1))
    if len(y_true) >= FGSM_advx_generator.samples:
        break

print(classification_report(y_true, y_pred, target_names=list(FGSM_advx_generator.class_indices.keys())))


In [ ]:
y_true, y_pred = [], []
for x_batch_raw, y_batch in PGD_advx_generator:
    x_in = preprocess_fn(tf.cast(x_batch_raw, tf.float32))
    preds = model.predict(x_in, verbose=0)
    y_true.extend(y_batch)
    y_pred.extend(np.argmax(preds, axis=1))
    if len(y_true) >= PGD_advx_generator.samples:
        break

print(classification_report(y_true, y_pred, target_names=list(PGD_advx_generator.class_indices.keys())))


In [ ]:
y_true, y_pred = [], []
for x_batch_raw, y_batch in raw_test_generator:
    x_in = preprocess_fn(tf.cast(x_batch_raw, tf.float32))
    preds = model.predict(x_in, verbose=0)
    y_true.extend(y_batch)
    y_pred.extend(np.argmax(preds, axis=1))
    if len(y_true) >= raw_test_generator.samples:
        break

print(classification_report(y_true, y_pred, target_names=list(raw_test_generator.class_indices.keys())))
